# LOB Visualization: Real vs Generated

Интерактивная визуализация эволюции книги заявок.
- Запустите ячейку 1 для **COND + REAL**
- Запустите ячейку 2 для **COND + GEN**

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

## Data Loading

In [2]:
# Use container path (mounted via Docker -v)
DATA_PATH = Path("/app/output/evalsequences/s5v2N5/GOOG/2023_Jan")
SAMPLE_IDS = [1490, 1498, 808, 907]

def load_all_data():
    """Load and concatenate all data. Returns cond_lens for junction marking."""
    real_books, gen_books = {}, {}
    real_msgs, gen_msgs = {}, {}
    cond_lens = {}  # Store conditioning sequence length for each sample
    
    for sid in SAMPLE_IDS:
        # Load conditioning data
        cond_book = np.loadtxt(DATA_PATH / f"data_cond/GOOG_2023-01-13_orderbook_real_id_{sid}.csv", delimiter=',')
        cond_msg = np.loadtxt(DATA_PATH / f"data_cond/GOOG_2023-01-13_message_real_id_{sid}.csv", delimiter=',')
        
        # Load real continuation
        real_book = np.loadtxt(DATA_PATH / f"data_real/GOOG_2023-01-13_orderbook_real_id_{sid}.csv", delimiter=',')
        real_msg = np.loadtxt(DATA_PATH / f"data_real/GOOG_2023-01-13_message_real_id_{sid}.csv", delimiter=',')
        
        # Load generated continuation
        gen_book = np.loadtxt(DATA_PATH / f"data_gen/GOOG_2023-01-13_orderbook_real_id_{sid}_gen_id_0.csv", delimiter=',')
        gen_msg = np.loadtxt(DATA_PATH / f"data_gen/GOOG_2023-01-13_message_real_id_{sid}_gen_id_0.csv", delimiter=',')
        
        # Store conditioning length (junction point)
        cond_lens[sid] = cond_book.shape[0]
        
        # Concatenate
        real_books[sid] = np.vstack([cond_book, real_book])
        gen_books[sid] = np.vstack([cond_book, gen_book])
        real_msgs[sid] = np.vstack([cond_msg, real_msg])
        gen_msgs[sid] = np.vstack([cond_msg, gen_msg])
        
        print(f"Sample {sid}: real={real_books[sid].shape}, gen={gen_books[sid].shape}, cond_len={cond_lens[sid]}")
    
    return real_books, gen_books, real_msgs, gen_msgs, cond_lens

real_books, gen_books, real_msgs, gen_msgs, cond_lens = load_all_data()

Sample 1490: real=(1001, 40), gen=(1001, 40), cond_len=501
Sample 1498: real=(1001, 40), gen=(1001, 40), cond_len=501
Sample 808: real=(1001, 40), gen=(1001, 40), cond_len=501
Sample 907: real=(1001, 40), gen=(1001, 40), cond_len=501


## Visualization Function

In [ ]:
def extract_quantities(book_row):
    """Extract quantities from orderbook row (LOBSTER interleaved format).
    Format: [ask_price1, ask_qty1, bid_price1, bid_qty1, ask_price2, ...]
    Returns: [bid_qty1, ..., bid_qty10, ask_qty1, ..., ask_qty10]
    """
    ask_qtys = book_row[1::4]   # indices 1, 5, 9, ... (10 values)
    bid_qtys = book_row[3::4]   # indices 3, 7, 11, ... (10 values)
    return np.concatenate([bid_qtys, ask_qtys])

def compute_queued_volumes(book_array):
    """Compute bid/ask/total queued volume over time."""
    T = book_array.shape[0]
    ask_vol = np.array([np.sum(np.abs(book_array[t, 1::4])) for t in range(T)])
    bid_vol = np.array([np.sum(np.abs(book_array[t, 3::4])) for t in range(T)])
    return bid_vol, ask_vol, bid_vol + ask_vol

def interactive_lob_plot(books, msgs, cond_lens, title="LOB", data_type="GEN"):
    """
    Interactive LOB visualization.
    
    Args:
        books: dict {sample_id: book_array}
        msgs: dict {sample_id: msg_array}
        cond_lens: dict {sample_id: conditioning_length} for junction marking
        title: title for the plot
        data_type: "REAL" or "GEN" - type of continuation data
    """
    
    # Controls
    id_dd = widgets.Dropdown(options=sorted(books.keys()), description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev = widgets.Button(description="←", layout=widgets.Layout(width="50px"))
    btn_next = widgets.Button(description="→", layout=widgets.Layout(width="50px"))
    btn_junction = widgets.Button(description="→ Cond/Gen", layout=widgets.Layout(width="100px"))
    
    # Separate display boxes
    time_box = widgets.HTML()      # Line 1: time
    sample_box = widgets.HTML()    # Line 2: sample info
    msg_box = widgets.HTML()       # Line 3: message
    vol_box = widgets.HTML()       # Line 4: volume
    
    # Figure: book panels + volumes
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[f"{title}: Book t-1", f"{title}: Book t", f"{title}: Volumes"],
        column_widths=[0.3, 0.3, 0.4]
    )
    fig.add_trace(go.Bar(x=[], y=[], name="t-1"), row=1, col=1)
    fig.add_trace(go.Bar(x=[], y=[], name="t"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Bid'), row=1, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Ask'), row=1, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Total'), row=1, col=3)
    # Current time cursor (gray dashed)
    fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref="x3", yref="paper",
                  line=dict(color="gray", width=1, dash="dash"))
    # Junction line (thick red solid)
    fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref="x3", yref="paper",
                  line=dict(color="red", width=3))
    fig.update_layout(width=1200, height=400, showlegend=False, template='plotly_white',
                      margin=dict(l=30, r=30, t=50, b=30))
    fig_widget = go.FigureWidget(fig)
    
    # Volume cache
    _vol_cache = {}
    
    def format_message(msg_row):
        """Format message: bold interpretation + raw array.
        Message format: [id, type, time, size, price, direction, ...]
        """
        m = msg_row.astype(int)
        msg_id = m[0]   # message/order id
        et = m[1]       # event_type
        ts = m[2]       # timestamp
        sz = m[3]       # size
        price = m[4]    # absolute price
        dr = m[5]       # direction (-1=Sell, 1=Buy)
        et_map = {1: "Limit", 2: "PartialCancel", 3: "Delete", 4: "Execution"}
        dr_map = {1: "Buy", -1: "Sell"}
        info = f"{et_map.get(et, '?')} • {dr_map.get(dr, '?')} • size={sz} • price={price}"
        return f"<b>{info}</b> | time={ts}<br>raw: {m.tolist()}"
    
    def update_sample(*_):
        sid = id_dd.value
        time_slider.min = 1
        time_slider.max = books[sid].shape[0] - 1
        time_slider.value = 1
        
        if sid not in _vol_cache:
            _vol_cache[sid] = compute_queued_volumes(books[sid])
        
        bid_v, ask_v, tot_v = _vol_cache[sid]
        xs = np.arange(len(bid_v))
        
        # Get junction position for this sample
        junction = cond_lens[sid]
        
        with fig_widget.batch_update():
            fig_widget.data[2].x = xs
            fig_widget.data[2].y = bid_v
            fig_widget.data[3].x = xs
            fig_widget.data[3].y = ask_v
            fig_widget.data[4].x = xs
            fig_widget.data[4].y = tot_v
            # Update junction red line position
            fig_widget.layout.shapes[1].x0 = junction
            fig_widget.layout.shapes[1].x1 = junction
        
        update_plot()
    
    def update_plot(*_):
        sid = id_dd.value
        t = time_slider.value
        arr = books[sid]
        msg_arr = msgs[sid]
        junction = cond_lens[sid]
        
        q0 = extract_quantities(arr[t - 1])
        q1 = extract_quantities(arr[t])
        
        # Bid (first 10) goes down (negative), Ask (last 10) goes up (positive)
        q0_signed = np.concatenate([-np.abs(q0[:10]), np.abs(q0[10:])])
        q1_signed = np.concatenate([-np.abs(q1[:10]), np.abs(q1[10:])])
        
        # Compute diff for coloring (on absolute values)
        diff = np.abs(q1) - np.abs(q0)
        x = np.arange(len(q0)) - len(q0) // 2
        # Gray for unchanged, red for increased, blue for decreased
        colors = ['gray' if abs(d) < 1e-8 else ('red' if d > 0 else 'blue') for d in diff]
        
        with fig_widget.batch_update():
            fig_widget.data[0].x = x
            fig_widget.data[0].y = q0_signed
            fig_widget.data[0].marker.color = 'gray'  # t-1 bars are gray
            fig_widget.data[1].x = x
            fig_widget.data[1].y = q1_signed
            fig_widget.data[1].marker.color = colors
            # Update current time cursor
            fig_widget.layout.shapes[0].x0 = t
            fig_widget.layout.shapes[0].x1 = t
            fig_widget.layout.annotations[0].text = f"{title}: Book t={t-1}"
            fig_widget.layout.annotations[1].text = f"{title}: Book t={t}"
        
        # Determine sample type (COND or REAL/GEN)
        if t < junction:
            sample_type = "COND"
        else:
            sample_type = data_type
        
        # Line 1: Time step display (big and clear)
        # time_box.value = f"<h4 style='margin:5px 0'>t = {t}</h4>"
        
        # Line 2: Sample info
        sample_box.value = f"Sample ID: {sid} | Segment: <b>{sample_type}</b> | junction @ {junction}"
        
        # Line 3: Message display
        msg_idx = t - 1
        if 0 <= msg_idx < len(msg_arr):
            msg_box.value = format_message(msg_arr[msg_idx])
        else:
            msg_box.value = "<b>Message:</b> (out of range)"
        
        # Line 4: Volume display (last)
        bid_v, ask_v, tot_v = _vol_cache[sid]
        vol_box.value = f"<b>Volume:</b> Bid={bid_v[t]:.0f} • Ask={ask_v[t]:.0f} • Total={tot_v[t]:.0f}"
    
    def on_prev(_):
        if time_slider.value > time_slider.min:
            time_slider.value -= 1
    
    def on_next(_):
        if time_slider.value < time_slider.max:
            time_slider.value += 1
    
    def on_junction(_):
        """Jump to the cond/gen junction point."""
        sid = id_dd.value
        junction = cond_lens[sid]
        if time_slider.min <= junction <= time_slider.max:
            time_slider.value = junction
    
    id_dd.observe(lambda _: update_sample(), names='value')
    time_slider.observe(lambda _: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_junction.on_click(on_junction)
    
    update_sample()
    
    controls = widgets.HBox([id_dd, btn_prev, btn_next, btn_junction, time_slider])
    display(widgets.HTML(f"<h3>{title}</h3>"))
    display(controls)
    display(fig_widget)
    display(time_box)
    display(sample_box)
    display(msg_box)
    display(vol_box)

## 1. COND + REAL

In [4]:
interactive_lob_plot(real_books, real_msgs, cond_lens, title="COND + REAL", data_type="REAL")

HTML(value='<h3>COND + REAL</h3>')

FigureWidget({
    'data': [{'marker': {'color': 'gray'},
              'name': 't-1',
              'type': 'bar',
              'uid': 'fe976922-0a72-4978-8508-3f0a7c3f6a80',
              'x': array([-10,  -9,  -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,
                            4,   5,   6,   7,   8,   9]),
              'xaxis': 'x',
              'y': array([ -100.,  -250.,  -774.,  -550., -1144.,  -350., -1061.,  -400., -1028.,
                           -900.,  1178.,   358.,   719.,   825.,  1106.,   726.,  1790.,   765.,
                            429.,    77.]),
              'yaxis': 'y'},
             {'marker': {'color': [gray, gray, gray, gray, gray, gray, gray, gray,
                                   blue, gray, gray, gray, gray, gray, gray, gray,
                                   gray, gray, gray, gray]},
              'name': 't',
              'type': 'bar',
              'uid': '7f839124-9b75-4e46-8b56-9fae1aff6e63',
              'x': array([-1

HTML(value="<h4 style='margin:5px 0'>t = 1</h4>")

HTML(value='Sample ID: 808 | Segment: <b>COND</b> | junction @ 501')

HTML(value='<b>Delete • Sell • size=100 • price=917500</b> | time=225367030<br>raw: [38571, 3, 225367030, 100,…

HTML(value='<b>Volume:</b> Bid=6457 • Ask=7973 • Total=14430')

## 2. COND + GEN

In [5]:
interactive_lob_plot(gen_books, gen_msgs, cond_lens, title="COND + GEN", data_type="GEN")

HTML(value='<h3>COND + GEN</h3>')

FigureWidget({
    'data': [{'marker': {'color': 'gray'},
              'name': 't-1',
              'type': 'bar',
              'uid': '2802303a-d9fd-4f73-9dfb-23705c00e9b2',
              'x': array([-10,  -9,  -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,
                            4,   5,   6,   7,   8,   9]),
              'xaxis': 'x',
              'y': array([ -100.,  -250.,  -774.,  -550., -1144.,  -350., -1061.,  -400., -1028.,
                           -900.,  1178.,   358.,   719.,   825.,  1106.,   726.,  1790.,   765.,
                            429.,    77.]),
              'yaxis': 'y'},
             {'marker': {'color': [gray, gray, gray, gray, gray, gray, gray, gray,
                                   blue, gray, gray, gray, gray, gray, gray, gray,
                                   gray, gray, gray, gray]},
              'name': 't',
              'type': 'bar',
              'uid': '5cd40067-8b4c-4f57-9aa7-4d71f217a988',
              'x': array([-1

HTML(value="<h4 style='margin:5px 0'>t = 1</h4>")

HTML(value='Sample ID: 808 | Segment: <b>COND</b> | junction @ 501')

HTML(value='<b>Delete • Sell • size=100 • price=917500</b> | time=225367030<br>raw: [38571, 3, 225367030, 100,…

HTML(value='<b>Volume:</b> Bid=6457 • Ask=7973 • Total=14430')

## 3. Midprice Comparison: REAL vs GEN

In [ ]:
# Colors for segments
COLOR_COND = '#888888'   # Gray for conditioning
COLOR_REAL = '#2E86AB'   # Blue for real
COLOR_GEN = '#A23B72'    # Purple/magenta for generated

def compute_midprice(book_array):
    """Compute midprice over time from LOBSTER orderbook data.
    Index 0 = best ask price, Index 2 = best bid price
    """
    return (book_array[:, 0] + book_array[:, 2]) / 2

def interactive_midprice_plot(real_books, gen_books, cond_lens):
    """Interactive midprice comparison: COND + REAL vs COND + GEN on one graph."""
    
    id_dd = widgets.Dropdown(options=sorted(real_books.keys()), description="Sample ID:")
    
    # Create figure
    fig = go.Figure()
    
    # Add placeholder traces
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='COND',
                             line=dict(color=COLOR_COND, width=2)))
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='REAL',
                             line=dict(color=COLOR_REAL, width=2)))
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='GEN',
                             line=dict(color=COLOR_GEN, width=2)))
    
    fig.update_layout(
        title='Midprice Evolution: REAL vs GEN',
        xaxis_title='Time Step',
        yaxis_title='Midprice',
        width=1000, height=400,
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    fig_widget = go.FigureWidget(fig)
    info_box = widgets.HTML()
    
    def update_plot(*_):
        sid = id_dd.value
        junction = cond_lens[sid]
        
        real_mid = compute_midprice(real_books[sid])
        gen_mid = compute_midprice(gen_books[sid])
        
        xs_cond = np.arange(junction + 1)
        xs_cont = np.arange(junction, len(real_mid))
        
        with fig_widget.batch_update():
            # COND segment (same for both)
            fig_widget.data[0].x = xs_cond
            fig_widget.data[0].y = real_mid[:junction + 1]
            
            # REAL continuation
            fig_widget.data[1].x = xs_cont
            fig_widget.data[1].y = real_mid[junction:]
            
            # GEN continuation
            fig_widget.data[2].x = xs_cont
            fig_widget.data[2].y = gen_mid[junction:]
            
            fig_widget.layout.title.text = f'Midprice Evolution - Sample {sid}'
        
        # Info
        real_final = real_mid[-1]
        gen_final = gen_mid[-1]
        cond_final = real_mid[junction]
        
        info_box.value = (
            f"<div style='font-family: monospace; font-size: 13px;'>"
            f"<b>Sample {sid}</b> | Junction @ {junction}<br>"
            f"Midprice at junction: <b>{cond_final:.0f}</b><br>"
            f"Final midprice: "
            f"<span style='color:{COLOR_REAL}'><b>REAL={real_final:.0f}</b></span> | "
            f"<span style='color:{COLOR_GEN}'><b>GEN={gen_final:.0f}</b></span> | "
            f"Diff={abs(real_final - gen_final):.0f}"
            f"</div>"
        )
    
    id_dd.observe(update_plot, names='value')
    update_plot()
    
    display(widgets.VBox([
        widgets.HTML("<h3>Midprice: REAL vs GEN</h3>"),
        id_dd,
        fig_widget,
        info_box
    ]))

interactive_midprice_plot(real_books, gen_books, cond_lens)